## Building a model to predict Warranty 'Component Description' from Transmission Test Data

Method: Use decision tree based model to predict a category of transmission-related warranty, given test data values. 

Inputs:
- `in/Unique Component Descriptions`, `sheet="Unique Component Descriptions"` for unique component descriptions
- `in/test_data_per_transmission.xlsx`, which holds each transmission's tests passed % for each test section. 
- `in/CLEANED_Full_Claims_Report.xlsx`, which is cleaned warranty data.



### Cleaning

uses cleaned data from 1b. 


In [16]:
import pandas as pd

unique_component_descriptions = pd.read_csv(r'in/Unique Component Descriptions.csv', encoding='unicode_escape').dropna(axis=1)
#unique_component_descriptions 

test_data_df = pd.read_csv(r'in/test_data_per_transmission.csv', encoding='unicode_escape')
#test_data_df 

warranty_df = pd.read_excel('in/CLEANED_Full_Claims_Report.xlsx', sheet_name='Sheet1')


We also separate test into two different prediction tasks: for CVT transmission and for PowerTrain transmissions, which are mutually exclusive transmission types and use a different set of tests.

Mike says only CVT tests will have a value for `CVT Condition X` so here we are just using `CVT Condition 1`.


In [17]:
test_data_cvt = test_data_df[test_data_df['CVT Condition 1'].isna() == False]
test_data_cvt.head()


,SerialNumber,1/2 Clutch,3/4 Clutch,5/6 Clutch,Brake OBT,C1 Clutch,C2 Clutch,C3 Clutch,C4 Clutch,CVT Condition 1,...,Remote Valves,Reverse 1,Reverse 2,Reverse 3,Reverse 4,Reverse Clutch,Service Brake Function,Steering Relief Pressure,Trailer Brake Release,Trailer Brake Relief Pressure
2,AJB06$3528,NaN,NaN,NaN,1.000000,0.875000,0.857143,0.833333,0.827586,1.000000,...,1.0,NaN,NaN,NaN,NaN,0.818182,1.0,1.000000,1.0,0.75
7,AJB0652038,NaN,NaN,NaN,0.727273,0.880000,1.000000,1.000000,0.941176,1.000000,...,1.0,NaN,NaN,NaN,NaN,0.531250,1.0,1.000000,1.0,1.00
9,AJB0652040,NaN,NaN,NaN,0.875000,0.884615,0.937500,0.933333,0.882353,0.777778,...,1.0,NaN,NaN,NaN,NaN,0.777778,1.0,0.428571,NaN,NaN
10,AJB0652044,NaN,NaN,NaN,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.0,NaN,NaN,NaN,NaN,0.833333,1.0,0.500000,NaN,NaN
11,AJB0652046,NaN,NaN,NaN,1.000000,0.900000,1.000000,1.000000,0.733333,1.000000,...,1.0,NaN,NaN,NaN,NaN,0.916667,1.0,0.600000,1.0,1.00



Examining the shape, we see the number of columns if we only drop columns with "all" NaN values vs if we drop columns with "any" NaN values.

In [18]:
print(test_data_cvt.dropna(axis=1, how='all').shape) # drop any columns missing EVERY value. (results in 47 columns)
print(test_data_cvt.dropna(axis=1).shape) # drop any column missing ANY value (will result in 19 columns)

(707, 47)
(707, 19)


In [19]:
# TODO: ask Mike his preference here
test_data_cvt = test_data_cvt.dropna(axis=1, how='all') # drop any columns missing EVERY value. (results in 47 columns)
#test_data_cvt = test_data_cvt.dropna(axis=1) # drop any column missing ANY value (will result in 19 columns)


Similarly PowerTrain Transmissions **won't** have a value for `CVT Condition 1`, so we split in a similar way.

In [20]:
test_data_powertrain = test_data_df[test_data_df['CVT Condition 1'].isna() == True]
test_data_powertrain.head()


,SerialNumber,1/2 Clutch,3/4 Clutch,5/6 Clutch,Brake OBT,C1 Clutch,C2 Clutch,C3 Clutch,C4 Clutch,CVT Condition 1,...,Remote Valves,Reverse 1,Reverse 2,Reverse 3,Reverse 4,Reverse Clutch,Service Brake Function,Steering Relief Pressure,Trailer Brake Release,Trailer Brake Relief Pressure
0,AJB06$3$24,0.8,0.8,0.8,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,1.0,0.8,1.0,1.0,0.8,1.0,1.000000,1.0,0.0
1,AJB06$3508,0.8,0.8,0.8,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,1.0,1.0,1.0,1.0,0.8,1.0,0.500000,1.0,0.0
3,AJB0650718,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.000000,NaN,NaN
4,AJB0650847,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.333333,NaN,NaN
5,AJB0652032,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Examining the shape, we see the number of columns if we only drop columns with "all" NaN values vs if we drop columns with "any" NaN values.

In [21]:
print(test_data_powertrain.dropna(axis=1, how="all").shape)
print(test_data_powertrain.dropna(axis=1, how="any").shape)

(1242, 69)
(1242, 1)


This indicates there are at least one missing value in every test. I suspect that there are some powertrain transmissions that didn't undergo every test-- we will see how to deal with this in sprint.

For now, just drop a column if all values are missing.

In [22]:
test_data_powertrain = test_data_powertrain.dropna(axis=1, how="all")

Get warranty data, assume anything in 'Unique Component Descriptions' with 'Is Transmission Related' == N is a pass.

In [23]:
warranty_df = warranty_df[['Transmission Number', 'Component Description']]
warranty_df = warranty_df[warranty_df['Transmission Number'] != '#']

transmission_related_df = unique_component_descriptions[unique_component_descriptions['Is Transmission Related'] == 'Y']
transmission_related = set(transmission_related_df['Component Description'])



SQL style joining of warranty data with test data, on Transmission Number

additionally, add a new column that will be the Value column

In [24]:
def join_with_warranty(test_data: pd.DataFrame, warranty_df=warranty_df) -> pd.DataFrame:
    '''
    LEFT outer join the inputted test_data with the warranty data.
    fills in 'Component Description' column

    we do a left outer join to keep any transmissions that have been tested but have no warranty, and get rid of 
    transmissions with a warranty but without test data.
    '''
    result = pd.merge(test_data, 
                    warranty_df, 
                    how="left", 
                    left_on="SerialNumber", 
                    right_on="Transmission Number")
    
    result['Value'] = result['Component Description'].where(result['Component Description'].isin(transmission_related), 'No Transmission Issue')

    return result.drop(columns=['Component Description', 'Transmission Number']) # drop some unneeded columns


cvt = join_with_warranty(test_data=test_data_cvt)
powertrain = join_with_warranty(test_data=test_data_powertrain)
cvt.to_excel('out/temp.xlsx')
cvt.head()


,SerialNumber,Brake OBT,C1 Clutch,C2 Clutch,C3 Clutch,C4 Clutch,CVT Condition 1,CVT Condition 10,CVT Condition 11,CVT Condition 12,...,Pressures @ High Speed,Pressures @ Low Speed,Pressures @ Mid Speed,Remote Valves,Reverse Clutch,Service Brake Function,Steering Relief Pressure,Trailer Brake Release,Trailer Brake Relief Pressure,Value
0,AJB06$3528,1.000000,0.875000,0.857143,0.833333,0.827586,1.000000,0.861111,0.904762,0.966667,...,1.0,1.0,1.000,1.0,0.818182,1.0,1.000000,1.0,0.75,No Transmission Issue
1,AJB0652038,0.727273,0.880000,1.000000,1.000000,0.941176,1.000000,1.000000,1.000000,1.000000,...,1.0,1.0,1.000,1.0,0.531250,1.0,1.000000,1.0,1.00,No Transmission Issue
2,AJB0652040,0.875000,0.884615,0.937500,0.933333,0.882353,0.777778,0.833333,0.833333,0.833333,...,1.0,1.0,1.000,1.0,0.777778,1.0,0.428571,NaN,NaN,No Transmission Issue
3,AJB0652044,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.0,1.0,1.000,1.0,0.833333,1.0,0.500000,NaN,NaN,No Transmission Issue
4,AJB0652046,1.000000,0.900000,1.000000,1.000000,0.733333,1.000000,1.000000,1.000000,1.000000,...,1.0,1.0,0.875,1.0,0.916667,1.0,0.600000,1.0,1.00,No Transmission Issue


In [25]:
def group_label(x):
    if x == "No Transmission Issue":
        return "No Issue"
    elif "valve" in x.lower() or "hydraulic" in x.lower():
        return "Hydraulic"
    elif "sensor" in x.lower() or "electronic" in x.lower():
        return "Electrical"
    elif "shaft" in x.lower() or "clutch" in x.lower():
        return "Mechanical"
    else:
        return "Other"


# for predicting the broader category
cvt["Value"] = cvt["Value"].apply(group_label)
powertrain["Value"] = powertrain["Value"].apply(group_label)

In [26]:
print(cvt.shape)
print(powertrain.shape)
print(cvt['Value'].value_counts())

(1072, 48)
(1501, 70)
Value
No Issue      762
Hydraulic     142
Other         116
Electrical     31
Mechanical     21
Name: count, dtype: int64


In [34]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report

def train(df):
    # we can't use anything where the component description only appears once, 
    # we can only use rows that have a value count of more than 1
    #counts = df['Value'].value_counts()
    #df = df[df['Value'].isin(counts[counts > 1].index)]

    X = df.drop(columns=['SerialNumber', 'Value'])
    y = df['Value']

    le = LabelEncoder()
    y = le.fit_transform(y)

    print('value counts')
    print(df['Value'].value_counts())

    # split into test and train
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    model = LGBMClassifier(
        n_estimators=200,   # reduced a bit bc overfitting
        learning_rate=0.05,
        num_leaves=15,      # smaller trees
        max_depth=5,        # smaller trees
        random_state=42,
        reg_alpha=1.0,           # L1 regularization
        reg_lambda=1.0,          # L2 regularization
        class_weight='balanced',
        verbosity=-1
    )

    model.fit(X_train, y_train)

    # evaluate the model
    y_pred = model.predict(X_test)
    print('classification report:\n', classification_report(y_test, y_pred))

    # cross validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=cv)
    print('cross validation scores, mean and std:', scores.mean(), scores.std())

    # feature importance
    importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)

    print(importance_df.head(10))


    

train(cvt)


value counts
Value
No Issue      762
Hydraulic     142
Other         116
Electrical     31
Mechanical     21
Name: count, dtype: int64
classification report:
               precision    recall  f1-score   support

           0       0.05      0.17      0.07         6
           1       0.27      0.28      0.27        29
           2       0.00      0.00      0.00         4
           3       0.92      0.64      0.75       153
           4       0.19      0.39      0.25        23

    accuracy                           0.54       215
   macro avg       0.28      0.29      0.27       215
weighted avg       0.71      0.54      0.60       215

cross validation scores, mean and std: 0.5055987828732885 0.030660899640585937
                     feature  importance
43  Steering Relief Pressure        1192
1                  C1 Clutch         749
36          Park Brake Decay         571
17           CVT Condition 5         511
4                  C4 Clutch         487
40             Remote Valve

In [28]:
train(powertrain)

value counts
Value
No Issue      1285
Hydraulic      101
Other           86
Electrical      25
Mechanical       4
Name: count, dtype: int64
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000663 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 910
[LightGBM] [Info] Number of data points in the train set: 1200, number of used features: 59
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM

c:\Users\79cap\miniconda3\envs\cnh\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [29]:
cvt.to_excel('out/cvt.xlsx')
powertrain.to_excel('out/powertrain.xlsx')